# DistilBERT Relation Classifier — Google Colab GPU notebook

This notebook runs only the transformer model for the Fandom character relationship extraction project. It expects the split files created by `01_data_pipeline.ipynb` to already exist in `data/`.

Run the local pipeline first, sync or upload the project folder to Google Drive as `MyDrive/fandom_kg_project`, then run this notebook in Colab with **Runtime → Change runtime type → GPU**. The notebook writes transformer outputs to `transformer/`, especially `transformer/predictions_all_transformer.csv` for graph generation.

## 0. Colab setup, Drive, and packages

Run this setup cell first. In Colab, it may restart the runtime once after installing a consistent NumPy / scikit-learn stack. After it reconnects, run the notebook again from the top.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = running_in_colab()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PROJECT_NAME = "fandom_kg_project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Running in Colab: {IN_COLAB}")
print(f"Project directory: {PROJECT_DIR}")


def pip_install(packages: list[str], *, force: bool = False) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if force:
        cmd.append("--force-reinstall")
    cmd.extend(packages)
    subprocess.check_call(cmd)


# Colab occasionally has a half-upgraded NumPy stack. That causes errors like:
# ImportError: cannot import name '_center' from 'numpy._core.umath'
# Install a consistent NumPy / pandas / scikit-learn stack first, then restart once.
if IN_COLAB:
    SETUP_STAMP = Path("/content/.distilbert_colab_deps_installed")
    if not SETUP_STAMP.exists():
        pip_install([
            "numpy==1.26.4",
            "scipy==1.13.1",
            "pandas==2.2.2",
            "scikit-learn==1.5.2",
        ], force=True)
        pip_install([
            "transformers==4.44.2",
            "accelerate==0.33.0",
            "huggingface_hub>=0.24.0",
            "sentencepiece>=0.2.0",
            "safetensors>=0.4.3",
        ], force=False)
        SETUP_STAMP.write_text("installed\n")
        print("Dependencies installed. Restarting runtime once; after reconnecting, run the notebook again from the top.")
        os.kill(os.getpid(), 9)
else:
    required_packages = [
        "pandas",
        "numpy",
        "scikit-learn",
        "torch",
        "transformers",
        "accelerate",
        "huggingface_hub",
        "sentencepiece",
        "safetensors",
    ]
    for package in required_packages:
        import_name = {
            "scikit-learn": "sklearn",
            "huggingface_hub": "huggingface_hub",
        }.get(package, package.replace("-", "_"))
        if importlib.util.find_spec(import_name) is None:
            pip_install([package])

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")

GPU_AVAILABLE = torch.cuda.is_available()
GPU_DEVICE = "cuda" if GPU_AVAILABLE else "cpu"
print(f"CUDA available: {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. In Colab, use Runtime -> Change runtime type -> GPU.")

## 1. Configuration

In [ ]:
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Inputs produced by 01_data_pipeline.ipynb.
CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
TRAIN_CSV = DATA_DIR / "train.csv"
DEV_CSV = DATA_DIR / "dev.csv"
TEST_CSV = DATA_DIR / "test.csv"
PIPELINE_RUN_METADATA_JSON = DATA_DIR / "pipeline_run_metadata.json"
LLM_LABELED_CANDIDATES_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"

# Transformer outputs. Copy this folder back into your local project after Colab finishes.
TRANSFORMER_DIR = PROJECT_DIR / "transformer"
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]
NO_RELATION_LABEL = "no_relation"

RANDOM_SEED = 42

# Training settings. Lower batch size if Colab runs out of memory.
TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"
TRANSFORMER_TEXT_COLUMN = "text_marked"
TRANSFORMER_MAX_LEN = 192
TRANSFORMER_BATCH_SIZE = 8 if GPU_AVAILABLE else 4
TRANSFORMER_EPOCHS = 4
TRANSFORMER_LEARNING_RATE = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
TRANSFORMER_WARMUP_RATIO = 0.10
TRANSFORMER_PATIENCE = 2
TRANSFORMER_GRAD_CLIP = 1.0
TRANSFORMER_SPECIAL_TOKENS = ["[HEAD]", "[/HEAD]", "[TAIL]", "[/TAIL]"]
REQUIRE_GPU_FOR_TRANSFORMER = True

print(f"Data folder: {DATA_DIR}")
print(f"Transformer output folder: {TRANSFORMER_DIR}")
print(f"Train CSV exists: {TRAIN_CSV.exists()}")
print(f"Dev CSV exists: {DEV_CSV.exists()}")
print(f"Test CSV exists: {TEST_CSV.exists()}")
print(f"Candidate CSV exists: {CANDIDATE_EXAMPLES_CSV.exists()}")
print(f"GPU device: {GPU_DEVICE}")

## 2. Imports and helper functions

In [ ]:
import copy
import json
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

pd.set_option("display.max_colwidth", 140)


def require_file(path: Path, upstream_notebook: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run {upstream_notebook} first or copy the file into Google Drive.")
    return path


def set_transformer_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def evaluate_predictions(y_true, y_pred, labels_order: list[str]) -> dict:
    labels_present = [label for label in labels_order if label in set(y_true) | set(y_pred)]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels_present, average="weighted", zero_division=0),
    }

## 3. Load split data

In [ ]:
if REQUIRE_GPU_FOR_TRANSFORMER and not GPU_AVAILABLE:
    raise RuntimeError(
        "REQUIRE_GPU_FOR_TRANSFORMER=True but no GPU was detected. "
        "In Colab, select Runtime -> Change runtime type -> GPU, then rerun the notebook."
    )

pipeline_metadata = json.loads(PIPELINE_RUN_METADATA_JSON.read_text(encoding="utf-8")) if PIPELINE_RUN_METADATA_JSON.exists() else {}

candidate_source_csv = Path(pipeline_metadata.get("candidate_source_csv", CANDIDATE_EXAMPLES_CSV))
if not candidate_source_csv.is_absolute():
    candidate_source_csv = PROJECT_DIR / candidate_source_csv
if not candidate_source_csv.exists() and LLM_LABELED_CANDIDATES_CSV.exists():
    candidate_source_csv = LLM_LABELED_CANDIDATES_CSV

LABEL_SOURCE = pipeline_metadata.get("label_source", "weak_labels")

candidates_df = pd.read_csv(require_file(candidate_source_csv, "01_data_pipeline.ipynb"))
train_df = pd.read_csv(require_file(TRAIN_CSV, "01_data_pipeline.ipynb"))
dev_df = pd.read_csv(require_file(DEV_CSV, "01_data_pipeline.ipynb"))
test_df = pd.read_csv(require_file(TEST_CSV, "01_data_pipeline.ipynb"))
split_method = pipeline_metadata.get("split_method", "loaded_from_csv")

required_columns = {
    "candidate_id",
    "head",
    "tail",
    "context",
    "label",
    "weak_label",
    "source_type",
    "pair_id",
    "text_basic",
    "text_marked",
}
for frame_name, frame in {
    "candidates_df": candidates_df,
    "train_df": train_df,
    "dev_df": dev_df,
    "test_df": test_df,
}.items():
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise ValueError(f"{frame_name} is missing required columns: {missing}")

for frame_name, frame in {"train_df": train_df, "dev_df": dev_df, "test_df": test_df}.items():
    invalid_labels = sorted(set(frame["label"].dropna().astype(str)) - set(RELATIONSHIPS))
    if invalid_labels:
        raise ValueError(f"{frame_name} contains labels not listed in RELATIONSHIPS: {invalid_labels}")

print(f"Loaded candidates: {len(candidates_df):,} rows ({LABEL_SOURCE}) from {candidate_source_csv}")
print(f"Loaded split: train={len(train_df):,}, dev={len(dev_df):,}, test={len(test_df):,}; split_method={split_method}")
display(train_df["label"].value_counts().rename_axis("label").reset_index(name="train_count"))

## 4. Build DistilBERT datasets and model

In [ ]:
TRANSFORMER_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if TRANSFORMER_DEVICE.type == "cpu":
    torch.set_num_threads(1)

print(f"Transformer device: {TRANSFORMER_DEVICE}")
print(f"Transformer base model: {TRANSFORMER_MODEL_NAME}")
print(f"Transformer batch size: {TRANSFORMER_BATCH_SIZE}")

set_transformer_seed(RANDOM_SEED)

transformer_label_to_id = {label: idx for idx, label in enumerate(RELATIONSHIPS)}
transformer_id_to_label = {idx: label for label, idx in transformer_label_to_id.items()}

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
num_added_tokens = tokenizer.add_special_tokens({"additional_special_tokens": TRANSFORMER_SPECIAL_TOKENS})
print(f"Added {num_added_tokens} entity-marker tokens to the tokenizer.")


class TransformerRelationDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        text_column: str,
        tokenizer,
        label_to_id: dict[str, int],
        max_len: int,
        include_labels: bool = True,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_column].fillna("").astype(str).tolist()
        self.tokenizer = tokenizer
        self.label_to_id = label_to_id
        self.max_len = max_len
        self.include_labels = include_labels

        if include_labels:
            missing_labels = sorted(set(self.df["label"]) - set(label_to_id))
            if missing_labels:
                raise ValueError(f"Found labels not present in RELATIONSHIPS: {missing_labels}")
            self.labels = self.df["label"].map(label_to_id).astype(int).tolist()
        else:
            self.labels = None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }

        if self.include_labels:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


train_transformer_dataset = TransformerRelationDataset(train_df, TRANSFORMER_TEXT_COLUMN, tokenizer, transformer_label_to_id, TRANSFORMER_MAX_LEN, include_labels=True)
dev_transformer_dataset = TransformerRelationDataset(dev_df, TRANSFORMER_TEXT_COLUMN, tokenizer, transformer_label_to_id, TRANSFORMER_MAX_LEN, include_labels=True)
test_transformer_dataset = TransformerRelationDataset(test_df, TRANSFORMER_TEXT_COLUMN, tokenizer, transformer_label_to_id, TRANSFORMER_MAX_LEN, include_labels=True)

train_transformer_loader = DataLoader(train_transformer_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True)
dev_transformer_loader = DataLoader(dev_transformer_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False)
test_transformer_loader = DataLoader(test_transformer_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False)


def make_transformer_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        TRANSFORMER_MODEL_NAME,
        num_labels=len(RELATIONSHIPS),
        id2label=transformer_id_to_label,
        label2id=transformer_label_to_id,
    )
    model.resize_token_embeddings(len(tokenizer))
    return model


transformer_model = make_transformer_model().to(TRANSFORMER_DEVICE)

train_label_counts = Counter(train_df["label"])
class_weights = []
for label in RELATIONSHIPS:
    count = max(1, int(train_label_counts.get(label, 0)))
    class_weights.append(len(train_df) / (len(RELATIONSHIPS) * count))
transformer_class_weights = torch.tensor(class_weights, dtype=torch.float32, device=TRANSFORMER_DEVICE)
transformer_loss_fn = torch.nn.CrossEntropyLoss(weight=transformer_class_weights)

optimizer = torch.optim.AdamW(transformer_model.parameters(), lr=TRANSFORMER_LEARNING_RATE, weight_decay=TRANSFORMER_WEIGHT_DECAY)
num_training_steps = max(1, len(train_transformer_loader) * TRANSFORMER_EPOCHS)
num_warmup_steps = int(num_training_steps * TRANSFORMER_WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps)


def run_transformer_prediction(model, loader: DataLoader):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(TRANSFORMER_DEVICE)
            attention_mask = batch["attention_mask"].to(TRANSFORMER_DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            pred_ids = torch.argmax(probs, dim=1)

            all_preds.extend([transformer_id_to_label[int(idx)] for idx in pred_ids.cpu().numpy()])
            all_probs.append(probs.cpu().numpy())

            if "labels" in batch:
                all_labels.extend([transformer_id_to_label[int(idx)] for idx in batch["labels"].cpu().numpy()])

    all_probs = np.vstack(all_probs) if all_probs else np.empty((0, len(RELATIONSHIPS)))
    return all_preds, all_probs, all_labels

## 5. Train with early stopping on dev macro-F1

In [ ]:
best_transformer_dev_macro_f1 = -1.0
best_transformer_state = None
epochs_without_improvement = 0
transformer_history_rows = []

for epoch in range(1, TRANSFORMER_EPOCHS + 1):
    epoch_start = time.time()
    transformer_model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in train_transformer_loader:
        input_ids = batch["input_ids"].to(TRANSFORMER_DEVICE)
        attention_mask = batch["attention_mask"].to(TRANSFORMER_DEVICE)
        labels = batch["labels"].to(TRANSFORMER_DEVICE)

        optimizer.zero_grad()
        outputs = transformer_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = transformer_loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), max_norm=TRANSFORMER_GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    train_loss = total_loss / max(1, total_examples)
    dev_pred, _, _ = run_transformer_prediction(transformer_model, dev_transformer_loader)
    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)

    transformer_history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_accuracy": dev_metrics["accuracy"],
            "dev_macro_f1": dev_metrics["macro_f1"],
            "dev_weighted_f1": dev_metrics["weighted_f1"],
            "epoch_seconds": time.time() - epoch_start,
        }
    )

    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
        f"dev_macro_f1={dev_metrics['macro_f1']:.4f} | "
        f"dev_accuracy={dev_metrics['accuracy']:.4f}"
    )

    if dev_metrics["macro_f1"] > best_transformer_dev_macro_f1:
        best_transformer_dev_macro_f1 = dev_metrics["macro_f1"]
        best_transformer_state = copy.deepcopy(transformer_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= TRANSFORMER_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

if best_transformer_state is not None:
    transformer_model.load_state_dict(best_transformer_state)

transformer_history_df = pd.DataFrame(transformer_history_rows)
transformer_history_df.to_csv(TRANSFORMER_DIR / "training_history.csv", index=False)
display(transformer_history_df)

## 6. Evaluate and save model artifacts

In [ ]:
dev_pred, dev_probs, _ = run_transformer_prediction(transformer_model, dev_transformer_loader)
test_pred, test_probs, _ = run_transformer_prediction(transformer_model, test_transformer_loader)

dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

transformer_metrics = {
    "variant": "distilbert_marked",
    "text_column": TRANSFORMER_TEXT_COLUMN,
    "base_model": TRANSFORMER_MODEL_NAME,
    "dev_accuracy": dev_metrics["accuracy"],
    "dev_macro_f1": dev_metrics["macro_f1"],
    "dev_weighted_f1": dev_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "train_examples": len(train_df),
    "dev_examples": len(dev_df),
    "test_examples": len(test_df),
    "max_len": TRANSFORMER_MAX_LEN,
    "batch_size": TRANSFORMER_BATCH_SIZE,
    "learning_rate": TRANSFORMER_LEARNING_RATE,
    "epochs_run": int(len(transformer_history_df)),
    "best_dev_macro_f1": float(best_transformer_dev_macro_f1),
}

transformer_metrics_df = pd.DataFrame([transformer_metrics])
transformer_metrics_df.to_csv(TRANSFORMER_DIR / "metrics_transformer.csv", index=False)

labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
transformer_report = classification_report(test_df["label"], test_pred, labels=labels_present, output_dict=True, zero_division=0)
pd.DataFrame(transformer_report).transpose().to_csv(TRANSFORMER_DIR / "classification_report_transformer.csv")

transformer_cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
transformer_cm_df = pd.DataFrame(
    transformer_cm,
    index=[f"true_{label}" for label in labels_present],
    columns=[f"pred_{label}" for label in labels_present],
)
transformer_cm_df.to_csv(TRANSFORMER_DIR / "confusion_matrix_transformer.csv")

transformer_test_out = test_df.copy()
transformer_test_out["predicted_label"] = test_pred
transformer_test_out["confidence"] = test_probs.max(axis=1) if len(test_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    transformer_test_out[f"prob_{label}"] = test_probs[:, label_idx] if len(test_probs) else []
transformer_test_out.to_csv(TRANSFORMER_DIR / "predictions_test_transformer.csv", index=False)

transformer_model_dir = TRANSFORMER_DIR / "distilbert_relation_classifier"
transformer_model_dir.mkdir(parents=True, exist_ok=True)
transformer_model.save_pretrained(transformer_model_dir)
tokenizer.save_pretrained(transformer_model_dir)

print(f"Saved DistilBERT evaluation outputs to: {TRANSFORMER_DIR}")
display(transformer_metrics_df)
display(transformer_cm_df)
display(transformer_test_out[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))

## 7. Predict all candidates for KG generation

In [ ]:
all_transformer_dataset = TransformerRelationDataset(
    candidates_df,
    TRANSFORMER_TEXT_COLUMN,
    tokenizer,
    transformer_label_to_id,
    TRANSFORMER_MAX_LEN,
    include_labels=False,
)
all_transformer_loader = DataLoader(all_transformer_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False)

transformer_all_pred = candidates_df.copy()
transformer_all_labels, transformer_all_probs, _ = run_transformer_prediction(transformer_model, all_transformer_loader)
transformer_all_pred["predicted_label"] = transformer_all_labels
transformer_all_pred["confidence"] = transformer_all_probs.max(axis=1) if len(transformer_all_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    transformer_all_pred[f"prob_{label}"] = transformer_all_probs[:, label_idx] if len(transformer_all_probs) else []

# Keep structured infobox relations as high-confidence weak labels only during weak-label runs.
# When LLM Judge labels are enabled, avoid reintroducing weak labels into KG generation.
if LABEL_SOURCE == "weak_labels":
    structured_mask = (
        (transformer_all_pred["source_type"] == "infobox")
        & (transformer_all_pred["weak_label"] != NO_RELATION_LABEL)
    )
    transformer_all_pred.loc[structured_mask, "predicted_label"] = transformer_all_pred.loc[structured_mask, "weak_label"]
    transformer_all_pred.loc[structured_mask, "confidence"] = 0.95
else:
    print("LLM Judge labels are enabled; skipping infobox weak-label override for DistilBERT KG predictions.")

transformer_all_predictions_path = TRANSFORMER_DIR / "predictions_all_transformer.csv"
transformer_all_pred.to_csv(transformer_all_predictions_path, index=False)

print(f"Saved all-candidate DistilBERT predictions to: {transformer_all_predictions_path}")
display(transformer_all_pred[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))

## 8. Output checklist

In [ ]:
expected_outputs = [
    TRANSFORMER_DIR / "metrics_transformer.csv",
    TRANSFORMER_DIR / "classification_report_transformer.csv",
    TRANSFORMER_DIR / "confusion_matrix_transformer.csv",
    TRANSFORMER_DIR / "predictions_test_transformer.csv",
    TRANSFORMER_DIR / "predictions_all_transformer.csv",
    TRANSFORMER_DIR / "training_history.csv",
    TRANSFORMER_DIR / "distilbert_relation_classifier",
]

print("DistilBERT notebook finished. Copy the transformer/ folder back into your local project if you trained in Colab.")
for path in expected_outputs:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")